In [ ]:
import time
import win32clipboard
import win32gui
import win32process
import win32api
import win32con
import psutil
import pyperclip
from pynput import keyboard

def get_active_window_info():
    """取得當前最前景視窗的標題與執行檔名稱"""
    hwnd = win32gui.GetForegroundWindow()
    if not hwnd:
        return "未知視窗", "未知程式"
    
    title = win32gui.GetWindowText(hwnd)
    try:
        _, pid = win32process.GetWindowThreadProcessId(hwnd)
        proc_name = psutil.Process(pid).name()
    except Exception:
        proc_name = "未知程式"
        
    return title, proc_name

def get_clipboard_content():
    """讀取剪貼簿內是檔案還是純文字"""
    copied_files = []
    try:
        win32clipboard.OpenClipboard()
        if win32clipboard.IsClipboardFormatAvailable(win32clipboard.CF_HDROP):
            data = win32clipboard.GetClipboardData(win32clipboard.CF_HDROP)
            if data:
                copied_files = list(data)
    except Exception:
        pass
    finally:
        try:
            win32clipboard.CloseClipboard()
        except Exception:
            pass

    if copied_files:
        return "FILES", copied_files
    else:
        return "TEXT", pyperclip.paste()

def send_key_combination(vk_code):
    """使用 Windows 原生 API 模擬按下 Ctrl + (C 或 V)"""
    # 釋放 Alt 鍵，避免影響 Ctrl+C/V
    win32api.keybd_event(win32con.VK_MENU, 0, win32con.KEYEVENTF_KEYUP, 0)
    time.sleep(0.05)
    
    # 按下 Ctrl -> 按下目標鍵 -> 釋放目標鍵 -> 釋放 Ctrl
    win32api.keybd_event(win32con.VK_CONTROL, 0, 0, 0)
    win32api.keybd_event(vk_code, 0, 0, 0)
    time.sleep(0.05)
    win32api.keybd_event(vk_code, 0, win32con.KEYEVENTF_KEYUP, 0)
    win32api.keybd_event(win32con.VK_CONTROL, 0, win32con.KEYEVENTF_KEYUP, 0)

def on_copy():
    """觸發 Ctrl + Alt + C"""
    print("\n" + "="*50)
    print("【偵測到 Ctrl + Alt + C：執行複製】")
    
    # 模擬標準 Ctrl+C
    send_key_combination(ord('C'))
    time.sleep(0.15)  # 等待剪貼簿寫入
    
    title, proc = get_active_window_info()
    content_type, content = get_clipboard_content()

    print(f"來源程式: {proc}")
    print(f"視窗標題: {title}")
    
    if content_type == "FILES":
        print(f"複製類型: 檔案/資料夾 (共 {len(content)} 個)")
        for idx, f in enumerate(content, 1):
            print(f"  {idx}. {f}")
    else:
        text_preview = content[:60] + ('...' if len(content) > 60 else '')
        print(f"複製類型: 純文字")
        print(f"複製內容: {text_preview}")
    print("="*50)

def on_paste():
    """觸發 Ctrl + Alt + V"""
    print("\n" + "="*50)
    print("【偵測到 Ctrl + Alt + V：執行貼上】")
    
    title, proc = get_active_window_info()
    content_type, content = get_clipboard_content()

    # 模擬標準 Ctrl+V
    send_key_combination(ord('V'))

    print(f"目標程式: {proc}")
    print(f"目標視窗: {title}")
    
    if content_type == "FILES":
        print(f"貼上類型: 檔案清單 ({len(content)} 個)")
    else:
        text_preview = content[:60] + ('...' if len(content) > 60 else '')
        print(f"貼上內容: {text_preview}")
    print("="*50)

def start_listener():
    print("【監聽啟動】請同時按住組合鍵：")
    print("  - Ctrl + Alt + C (複製並印出資訊)")
    print("  - Ctrl + Alt + V (貼上並印出資訊)")
    print("按 Ctrl + C (在命令提示字元視窗內) 可停止腳本\n")

    # 使用 GlobalHotKeys 原生組合鍵監聽，穩定度最高
    hotkeys = {
        '<ctrl>+<alt>+c': on_copy,
        '<ctrl>+<alt>+v': on_paste
    }

    with keyboard.GlobalHotKeys(hotkeys) as h:
        h.join()

if __name__ == "__main__":
    start_listener()

【監聽啟動】請同時按住組合鍵：
  - Ctrl + Alt + C (複製並印出資訊)
  - Ctrl + Alt + V (貼上並印出資訊)
按 Ctrl + C (在命令提示字元視窗內) 可停止腳本


【偵測到 Ctrl + Alt + C：執行複製】
來源程式: Code.exe
視窗標題: app-test.ipynb - 2.programm2 - Visual Studio Code - 已修改
複製類型: 純文字
複製內容: '<ctrl>+<alt>+v': on_paste
    }

    with keyboard.Globa...

【偵測到 Ctrl + Alt + V：執行貼上】
目標程式: brave.exe
目標視窗: Windows Server 跨 Wi-Fi 固定 IP 連線設定 - Google Gemini - Brave
貼上內容: '<ctrl>+<alt>+v': on_paste
    }

    with keyboard.Globa...

【偵測到 Ctrl + Alt + C：執行複製】
來源程式: Code.exe
視窗標題: app-test.ipynb - 2.programm2 - Visual Studio Code - 已修改
複製類型: 純文字
複製內容: lobalHotKeys 原生組合鍵監聽，穩定度最高
    hotkeys = {
        '<ctrl>...


In [ ]:
from presidio_analyzer import (
    AnalyzerEngine, 
    EntityRecognizer, 
    RecognizerResult,
    PatternRecognizer,
    Pattern
)
from presidio_analyzer.nlp_engine import NlpEngineProvider
from transformers import pipeline

# ---------------------------------------------------------------------------
# 1. 建立 Presidio 的基礎中文 NLP 引擎 (解決 KeyError: 'zh')
# ---------------------------------------------------------------------------
# 告訴 Presidio 中文 ('zh') 語系使用 spaCy 的 zh_core_web_sm (如果沒有安裝會自動警告，預設可載入)
nlp_configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "zh", "model_name": "zh_core_web_sm"},
        {"lang_code": "en", "model_name": "en_core_web_sm"}
    ],
}

provider = NlpEngineProvider(nlp_configuration=nlp_configuration)
custom_nlp_engine = provider.create_engine()

# ---------------------------------------------------------------------------
# 2. 自訂 Hugging Face 中文 NER 辨識器 (改用公開穩定的 ckiplab 模型)
# ---------------------------------------------------------------------------
class CustomHfChineseRecognizer(EntityRecognizer):
    def __init__(self):
        super().__init__(
            supported_entities=["PERSON", "LOCATION", "ORGANIZATION"],
            supported_language="zh"
        )
        print("正在載入 Hugging Face 中文 NER 模型 (ckiplab/bert-base-chinese-ner)...")
        
        # 改用確定公開可下載的 CKIP 繁體/通用中文 NER 模型
        self.ner_pipeline = pipeline(
            "ner", 
            model="ckiplab/bert-base-chinese-ner", 
            aggregation_strategy="simple"
        )

    def load(self):
        pass

    def analyze(self, text, entities, nlp_artifacts=None):
        results = []
        ner_results = self.ner_pipeline(text)
        
        # ckiplab 的標籤對映：PER -> PERSON, LOC -> LOCATION, ORG -> ORGANIZATION
        label_map = {
            "PER": "PERSON", 
            "LOC": "LOCATION", 
            "ORG": "ORGANIZATION"
        }

        for item in ner_results:
            entity_type = label_map.get(item["entity_group"])
            if entity_type and (not entities or entity_type in entities):
                results.append(
                    RecognizerResult(
                        entity_type=entity_type,
                        start=item["start"],
                        end=item["end"],
                        score=float(item["score"])
                    )
                )
        return results
# ---------------------------------------------------------------------------
# 3. 初始化 AnalyzerEngine 並註冊組件
# ---------------------------------------------------------------------------
# 傳入剛剛配置好的 custom_nlp_engine，並明確聲明支援 ["zh", "en"]
analyzer = AnalyzerEngine(
    nlp_engine=custom_nlp_engine, 
    supported_languages=["zh", "en"]
)

# 掛載自訂的 Hugging Face 模型
analyzer.registry.add_recognizer(CustomHfChineseRecognizer())


# ---------------------------------------------------------------------------
# 4. 測試執行
# ---------------------------------------------------------------------------
text = "你好，我是張偉，我的信箱是 zhangwei@example.com，我在台積電上班。"
results = analyzer.analyze(text=text, language="zh")

print("\n--- 檢測結果 ---")
for res in results:
    print(f"實體: {res.entity_type:<15} | 文字: {text[res.start:res.end]:<15} | 信心度: {res.score:.2f}")

正在載入 Hugging Face 中文 NER 模型 (ckiplab/bert-base-chinese-ner)...


Device set to use cpu



--- 檢測結果 ---
實體: EMAIL_ADDRESS   | 文字: zhangwei@example.com | 信心度: 1.00
實體: ORGANIZATION    | 文字: 台積              | 信心度: 1.00
實體: ORGANIZATION    | 文字: 電               | 信心度: 1.00
實體: LOCATION        | 文字: 台               | 信心度: 0.85
實體: URL             | 文字: example.com     | 信心度: 0.50
